# **LABORATORIO DIRIGIDO N° 04**
## **Fundamentos de Gestión de Datos**
### Semana 4 — Normalización de Bases de Datos y Consultas SQL Básicas

---

**Docente:** Pilar Rocío Sayán Mejía &nbsp;|&nbsp; **Semestre:** 2026-I &nbsp;|&nbsp; **Modalidad:** Individual — 2 horas

**Entregable:** Notebook `.ipynb` ejecutado con código, resultados visibles y respuestas completas

---

| Campo | Completa aquí |
|---|---|
| Nombre completo | |
| Sección | |
| Fecha | |

---
## I. Elemento de la Capacidad Terminal

Identifica violaciones a la Primera, Segunda y Tercera Forma Normal en una tabla desnormalizada, interpreta el modelo Entidad-Relación de la panadería Dulce Migaja e implementa consultas SQL básicas sobre la base de datos normalizada en SQLite, relacionando el diseño relacional con decisiones de negocio reales.

---
## Caso de Negocio: La Panadería Dulce Migaja

**Dulce Migaja** es una pequeña cadena de panaderías artesanales con varios locales en la ciudad. Durante años, el dueño registró todas sus ventas en una sola tabla llamada `ventas_original`. En esa tabla, cada fila repetía el nombre completo del cliente, el nombre del producto, la categoría y el local donde se realizó la venta.

Por ejemplo, si el cliente Carlos Quispe compraba pan de molde tres veces en el mismo mes, su nombre y teléfono aparecían repetidos en tres filas distintas. Cuando el dueño quería corregir un dato —como el nombre de un local— tenía que buscar y actualizar cada fila manualmente, lo que generaba errores e inconsistencias.

Para resolver este problema, el equipo normalizó la base de datos separando la información en tablas independientes: `categorias`, `productos`, `clientes`, `locales` y `ventas`. Ahora cada dato se guarda una sola vez y las tablas se conectan mediante claves foráneas.

> **Tu rol:** Eres el analista de datos de Dulce Migaja. Primero explorarás la tabla original para entender el problema, luego trabajarás con las tablas normalizadas para ver la solución y responderás preguntas del equipo directivo.

---
## **ACTIVIDAD 1: Revisión de Conceptos**

> Completa la siguiente tabla con **tus propias palabras**. No copies textualmente del material de clase. Usa el caso Dulce Migaja como referencia para los ejemplos.

| Concepto | Definición con tus palabras | Ejemplo del caso Dulce Migaja |
|---|---|---|
| Primera Forma Normal (1FN) | | |
| Segunda Forma Normal (2FN) | | |
| Tercera Forma Normal (3FN) | | |
| Dependencia parcial | | |
| Dependencia transitiva | | |
| Clave Primaria (PK) | | |
| Clave Foránea (FK) | | |
| Cardinalidad 1:N | | |
| Redundancia de datos | | |

**Responde las siguientes preguntas con tus propias palabras:**

**Pregunta 1:**
¿Qué es la normalización y qué tres problemas concretos resuelve en la panadería Dulce Migaja?

Respuesta:
_______________________________________________________________________

**Pregunta 2:**
¿Cuál es la diferencia entre una clave primaria (PK) y una clave foránea (FK)? Da un ejemplo con dos tablas de Dulce Migaja.

Respuesta:
_______________________________________________________________________

**Pregunta 3:**
En la tabla `ventas_original`, el nombre y teléfono del cliente se repiten en cada fila de venta. ¿Qué forma normal se viola? ¿Qué ocurre si el cliente cambia su número de teléfono?

Respuesta:
_______________________________________________________________________

**Pregunta 4:**
¿Por qué en el modelo normalizado la tabla `ventas` solo guarda `id_cliente` en lugar del nombre completo del cliente?

Respuesta:
_______________________________________________________________________

---
## **ACTIVIDAD 2: Exploración con SQLite**

En esta actividad explorarás la base de datos real de Dulce Migaja. La base de datos se descarga automáticamente desde GitHub. Ejecuta cada celda en orden y responde las preguntas después de observar el resultado.

### Relación entre las tablas

Antes de ejecutar las consultas, observa cómo se relacionan las tablas normalizadas:

```
clientes (id_cliente PK) ──────────────────┐
                                             │
productos (id_producto PK) ──────────────── ventas (id_venta PK)
                                             │
locales (id_local PK) ─────────────────────┘
    │
    └── categorias (id_categoria PK) → productos.id_categoria FK
```

**¿Por qué existen estas tablas separadas?** Cada entidad tiene su propia tabla para que sus datos se guarden una sola vez. Si cambia el nombre de un local, se actualiza en un solo lugar — no en cientos de filas de ventas.

### **Paso 0: Descargar la base de datos desde GitHub y conectarse**

Ejecuta esta celda primero. Descarga la base de datos de Dulce Migaja directamente desde el repositorio de GitHub y abre la conexión con SQLite.

In [ ]:
import sqlite3
import pandas as pd
import subprocess
from pathlib import Path

# Descargar la base de datos desde GitHub
url = (
    'https://raw.githubusercontent.com/Rociosayan/'
    'PMD1_Fundamentos_Gestion_Datos/main/casos/'
    '17_panaderia_normalizacion/panaderia_normalizacion.db'
)
db_path = Path('panaderia_normalizacion.db')

# Se usa curl para evitar problemas de certificados SSL en algunas instalaciones locales de Python.
comando = ['curl', '--ssl-no-revoke', '-L', '--fail', '-o', str(db_path), url]
resultado = subprocess.run(comando, capture_output=True, text=True)

# En algunos entornos Linux/Colab, curl puede no necesitar --ssl-no-revoke.
if resultado.returncode != 0:
    comando = ['curl', '-L', '--fail', '-o', str(db_path), url]
    resultado = subprocess.run(comando, capture_output=True, text=True)

if resultado.returncode != 0:
    raise RuntimeError('No se pudo descargar la base de datos desde GitHub. Revisa tu conexi?n a Internet.')

if not db_path.exists() or db_path.stat().st_size == 0:
    raise RuntimeError('La base de datos se descarg? vac?a. Vuelve a ejecutar la celda.')

# Conectarse a la base de datos SQLite
conn = sqlite3.connect(db_path)
print('Base de datos de Dulce Migaja cargada correctamente.')

### **Paso 0.1: ?Qu? contiene la base de datos?**

Antes de analizar la normalizaci?n, primero observa qu? tablas existen en la base de datos y qu? informaci?n guarda cada una. Este paso permite reconocer el contenido antes de interpretar las consultas.


In [ ]:
# Ver las tablas disponibles en la base de datos
tablas = pd.read_sql_query(
    '''
    SELECT name AS tabla
    FROM sqlite_schema
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    ''',
    conn
)

display(tablas)

### **Paso 0.2: Vista r?pida de cada tabla**

Ejecuta estas celdas para observar algunos registros de cada tabla. No necesitas modificar el c?digo; solo revisa qu? datos aparecen.


In [ ]:
# Tabla original desnormalizada
pd.read_sql_query('SELECT * FROM ventas_original LIMIT 5', conn)

In [ ]:
# Tablas normalizadas: categorias y productos
display(pd.read_sql_query('SELECT * FROM categorias', conn))
display(pd.read_sql_query('SELECT * FROM productos LIMIT 5', conn))

In [ ]:
# Tablas normalizadas: clientes, locales y ventas
display(pd.read_sql_query('SELECT * FROM clientes LIMIT 5', conn))
display(pd.read_sql_query('SELECT * FROM locales', conn))
display(pd.read_sql_query('SELECT * FROM ventas LIMIT 5', conn))

### **Paso 1: La tabla original — así guardaba Dulce Migaja sus ventas**

Antes de la normalización, todos los datos de cada venta se almacenaban en una sola tabla. Observa qué columnas tiene y qué información se repite.

In [ ]:
# Ver los primeros registros de la tabla original desnormalizada
df_original = pd.read_sql_query(
    'SELECT * FROM ventas_original LIMIT 15',
    conn
)
display(df_original)

**Pregunta 1:**
¿Qué columnas de `ventas_original` se repiten en varias filas? Menciona dos ejemplos concretos y explica qué problema genera esa repetición para el negocio.

Respuesta:
_______________________________________________________________________

### **Paso 2: ¿Cuántas veces se repite la información?**

Para ver la magnitud del problema, cuenta cuántas veces aparece cada cliente y cada categoría en la tabla original. Esto muestra exactamente cuántos registros habría que actualizar si cambia un dato.

In [ ]:
# Cuantas veces se repite cada cliente en la tabla original
df_rep_cli = pd.read_sql_query(
    '''
    SELECT cliente_nombre, COUNT(*) AS veces_repetido
    FROM ventas_original
    GROUP BY cliente_nombre
    ORDER BY veces_repetido DESC
    LIMIT 10
    ''',
    conn
)
display(df_rep_cli)

# Cuantas veces se repite cada categoria
df_rep_cat = pd.read_sql_query(
    '''
    SELECT categoria_nombre, COUNT(*) AS veces_repetido
    FROM ventas_original
    GROUP BY categoria_nombre
    ORDER BY veces_repetido DESC
    ''',
    conn
)
display(df_rep_cat)

**Pregunta 2:**
¿Cuántas veces aparece el cliente más repetido? Si ese cliente cambiara su número de teléfono, ¿cuántos registros tendría que actualizar el administrador en la tabla original? ¿Qué soluciona la normalización?

Respuesta:
_______________________________________________________________________

### **Paso 3: Las tablas normalizadas — categorias y productos**

Ahora observa cómo quedaron los datos después de la normalización. La tabla `categorias` guarda cada categoría una sola vez (cumple **1FN** y **3FN**). La tabla `productos` usa `id_categoria` como clave foránea en lugar de repetir el texto de la categoría (elimina dependencia transitiva → **3FN**).

In [ ]:
# Tabla de categorias: cada categoria una sola vez
df_cat = pd.read_sql_query('SELECT * FROM categorias', conn)
display(df_cat)

In [ ]:
# Tabla de productos: usa id_categoria como FK en vez de texto repetido
df_prod = pd.read_sql_query('SELECT * FROM productos', conn)
display(df_prod)

**Pregunta 3:**
¿Cuántas categorías distintas existen? ¿Qué columna de `productos` actúa como clave foránea? Si cambia el nombre de una categoría, ¿en cuántos lugares habría que actualizarlo con este diseño normalizado?

Respuesta:
_______________________________________________________________________

### **Paso 4: Clientes, locales y la tabla ventas normalizada**

Las tablas `clientes` y `locales` guardan cada entidad una sola vez (**1FN** y **2FN**). La tabla `ventas` normalizada ya no repite nombres: solo almacena `id_cliente`, `id_producto` e `id_local`. Esto cumple la **2FN** porque todos los atributos dependen completamente de `id_venta` (la clave primaria).

In [ ]:
# Tabla de clientes
df_cli = pd.read_sql_query('SELECT * FROM clientes', conn)
display(df_cli)

In [ ]:
# Tabla de locales
df_loc = pd.read_sql_query('SELECT * FROM locales', conn)
display(df_loc)

In [ ]:
# Tabla de ventas normalizada
df_ven = pd.read_sql_query('SELECT * FROM ventas LIMIT 10', conn)
display(df_ven)

**Pregunta 4:**
¿Qué diferencia principal observas entre `ventas_original` y `ventas`? ¿Por qué la tabla `ventas` normalizada solo contiene números (IDs) en lugar de nombres? ¿Qué forma normal garantiza esto?

Respuesta:
_______________________________________________________________________

### **Paso 5: ¿Qué producto tiene el precio más alto?**

El dueño quiere revisar los productos ordenados por precio para analizar su política de precios.

In [ ]:
# Productos ordenados de mayor a menor precio
df_precio = pd.read_sql_query(
    '''
    SELECT nombre AS producto, precio_unitario
    FROM productos
    ORDER BY precio_unitario DESC
    ''',
    conn
)
display(df_precio)

**Pregunta 5:**
¿Qué producto tiene el mayor precio unitario? ¿Y el menor? ¿Qué diferencia hay entre ambos precios?

Respuesta:
_______________________________________________________________________

### **Paso 6: ¿Cuántas ventas tiene cada categoría?**

La administradora quiere saber qué categoría concentra más ventas para decidir qué producir más.

In [ ]:
# Ventas totales por categoria (usando la tabla original para ver el dato directo)
df_cat_ven = pd.read_sql_query(
    '''
    SELECT categoria_nombre, COUNT(*) AS total_ventas
    FROM ventas_original
    GROUP BY categoria_nombre
    ORDER BY total_ventas DESC
    ''',
    conn
)
display(df_cat_ven)

**Pregunta 6:**
¿Qué categoría tiene más ventas registradas? ¿Qué recomendarías al equipo de producción basándote en este resultado?

Respuesta:
_______________________________________________________________________

### **Paso 7: ¿Qué local registra más ventas?**

El gerente quiere identificar el local con mayor actividad para reforzarlo con más personal. Aquí se usa JOIN para obtener el nombre del local en lugar del ID.

In [ ]:
# Ventas por local: se usa JOIN para obtener el nombre del local
df_loc_ven = pd.read_sql_query(
    '''
    SELECT l.nombre AS local, COUNT(*) AS total_ventas
    FROM ventas v
    INNER JOIN locales l ON v.id_local = l.id_local
    GROUP BY l.id_local, l.nombre
    ORDER BY total_ventas DESC
    ''',
    conn
)
display(df_loc_ven)

**Pregunta 7:**
¿Qué local registra más ventas? ¿Por qué fue necesario usar JOIN en lugar de consultar solo la tabla `ventas`?

Respuesta:
_______________________________________________________________________

### **Paso 8: Reporte completo de ventas con JOIN**

Para obtener un reporte legible — con nombres en lugar de números — se combinan las tablas normalizadas. Esto es exactamente lo que hace el JOIN: conecta las tablas usando las claves foráneas.

In [ ]:
# Reporte completo: nombre del cliente, producto, local y monto
df_reporte = pd.read_sql_query(
    '''
    SELECT
        v.id_venta,
        c.nombre AS cliente,
        p.nombre AS producto,
        l.nombre AS local,
        v.cantidad,
        p.precio_unitario,
        ROUND(v.cantidad * p.precio_unitario, 2) AS total_venta
    FROM ventas v
    INNER JOIN clientes  c ON v.id_cliente  = c.id_cliente
    INNER JOIN productos p ON v.id_producto = p.id_producto
    INNER JOIN locales   l ON v.id_local    = l.id_local
    ORDER BY total_venta DESC
    LIMIT 12
    ''',
    conn
)
display(df_reporte)

**Pregunta 8:**
¿Por qué fue necesario usar tres JOIN para generar este reporte? ¿Qué ventaja tiene hacer JOIN sobre una base normalizada frente a guardar todo en `ventas_original`?

Respuesta:
_______________________________________________________________________

### **Paso 9: ¿Qué cliente compra con más frecuencia?**

Marketing quiere identificar a los clientes más frecuentes para ofrecerles una tarjeta de fidelización.

In [ ]:
# Top 5 clientes con mas compras
df_top_cli = pd.read_sql_query(
    '''
    SELECT c.nombre AS cliente, COUNT(*) AS total_compras
    FROM ventas v
    INNER JOIN clientes c ON v.id_cliente = c.id_cliente
    GROUP BY c.id_cliente, c.nombre
    ORDER BY total_compras DESC
    LIMIT 5
    ''',
    conn
)
display(df_top_cli)

**Pregunta 9:**
¿Quién es el cliente más frecuente? ¿Qué acción concreta de marketing podría implementar Dulce Migaja para retenerlo?

Respuesta:
_______________________________________________________________________

### **Paso 10: ¿Qué producto genera más ingresos?**

El dueño quiere saber qué productos son los más rentables en términos de ingresos totales.

In [ ]:
# Ingresos totales por producto
df_ingresos = pd.read_sql_query(
    '''
    SELECT
        p.nombre AS producto,
        COUNT(*) AS veces_vendido,
        SUM(v.cantidad) AS unidades_totales,
        ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS ingresos_totales
    FROM ventas v
    INNER JOIN productos p ON v.id_producto = p.id_producto
    GROUP BY p.id_producto, p.nombre
    ORDER BY ingresos_totales DESC
    ''',
    conn
)
display(df_ingresos)

**Pregunta 10:**
¿Qué producto genera más ingresos totales? ¿Coincide con el producto más caro del Paso 5? ¿Qué conclusión puedes extraer sobre precio vs. volumen de ventas?

Respuesta:
_______________________________________________________________________

---
## **ACTIVIDAD 3: Aplicación y Reflexión**

Responde las siguientes preguntas **sin ejecutar código**. Usa tu comprensión del modelo normalizado de Dulce Migaja y los resultados obtenidos en la Actividad 2. Escribe respuestas en párrafos completos, no en listas.

---

**Situación 1:**
El dueño de Dulce Migaja decide que cada producto puede pertenecer a **más de una categoría** (por ejemplo, 'Pan de chocolate' podría ser 'Panadería' y 'Postres' al mismo tiempo).

¿La estructura actual de las tablas `productos` y `categorias` soporta este cambio? ¿Qué forma normal se vería afectada? ¿Qué tabla nueva habría que crear?

Respuesta:
_______________________________________________________________________

**Situación 2:**
Un desarrollador nuevo propone agregar directamente las columnas `nombre_cliente`, `telefono_cliente` y `direccion_cliente` a la tabla `ventas` para evitar el JOIN.

¿Qué forma normal se violaría con esta decisión? ¿Qué problemas concretos generaría para Dulce Migaja cuando un cliente cambie su dirección?

Respuesta:
_______________________________________________________________________

**Situación 3:**
La panadería quiere expandirse y comenzar a ofrecer **delivery**. Para eso necesita registrar: dirección de entrega, hora estimada de llegada, nombre del repartidor y costo del envío.

¿La estructura actual soporta este nuevo requerimiento? ¿Qué tabla o tablas nuevas crearías? Describe brevemente las columnas que tendría cada tabla nueva.

Respuesta:
_______________________________________________________________________

**Situación 4:**
La tabla `productos` tiene una columna llamada `id_categoria` (clave foránea). Imagina que en cambio hubiera una columna `nombre_categoria` de tipo texto (como en `ventas_original`).

¿Qué dependencia transitiva existiría? ¿Qué forma normal se violaría? ¿Qué pasaría si se cambiara el nombre de una categoría?

Respuesta:
_______________________________________________________________________

**Situación 5 — Integración:**
Basándote en todo lo explorado en este laboratorio, explica con tus propias palabras por qué Dulce Migaja tomó la decisión correcta al normalizar su base de datos. Menciona al menos **tres beneficios concretos** que obtiene el negocio con el diseño normalizado frente a la tabla `ventas_original`.

Respuesta:
_______________________________________________________________________

---
## **Conclusiones**

Escribe cinco conclusiones sobre lo aprendido en este laboratorio:

**1.** _¿Qué problema concreto resuelve la 1FN, la 2FN y la 3FN?_

**2.** _¿Qué diferencia fundamental existe entre la tabla `ventas_original` y el diseño normalizado?_

**3.** _¿Para qué sirve el JOIN y en qué situaciones lo usarías en el negocio de Dulce Migaja?_

**4.** _¿Qué consulta SQL de la Actividad 2 te resultó más útil para la toma de decisiones?_

**5.** _¿Por qué es más fácil mantener y actualizar una base de datos normalizada?_

In [ ]:
conn.close()
print('Conexion cerrada correctamente.')

---
**© 2026 - Fundamentos de Gestión de Datos - TECSUP**  
**Docente:** Pilar Rocío Sayán Mejía